[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-07-retry-timeout.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Resilience: @retry, @timeout, and @catch
**certified-journeys / metaflow-certified** · Practice · Fault-tolerant ML pipelines

> **Goal for today:** Make your Metaflow flows production-ready by adding automatic retries, execution timeouts, and graceful exception handling so transient failures never kill a pipeline run.


In [ ]:
%pip install -q metaflow


## Step 1 · Why resilience decorators exist

In production, pipelines fail for reasons beyond your control: network hiccups, throttled APIs, flaky cloud storage. Metaflow's resilience decorators let you describe *policy* — not retry logic — directly on each step.

| Decorator | Purpose | When to use |
|---|---|---|
| `@retry(times=N)` | Retry a failed step up to N extra times | External API calls, data downloads |
| `@timeout(seconds=S)` | Kill a step if it runs longer than S seconds | Steps with unbounded loops or blocking I/O |
| `@catch(var="e")` | Capture exception into an artifact, continue flow | Optional enrichment steps |

These decorators compose freely — you can stack `@retry` + `@timeout` on the same step.


In [ ]:
# Write a flow that demonstrates @retry with a simulated transient failure
flow_code = '''
import random
from metaflow import FlowSpec, step, retry

class RetryDemoFlow(FlowSpec):
    """Demonstrates @retry recovering from a simulated transient failure."""

    @step
    def start(self):
        # Seed a shared attempt counter via an artifact
        self.attempt_log = []
        self.next(self.flaky_download)

    @retry(times=3)  # Retry up to 3 times on failure (4 total attempts)
    @step
    def flaky_download(self):
        """Simulates a step that fails ~60% of the time — like a flaky API."""
        import os
        # Metaflow sets METAFLOW_CURRENT_RETRY_COUNT on retry attempts
        attempt = int(os.environ.get("METAFLOW_CURRENT_RETRY_COUNT", 0))
        print(f"  [flaky_download] attempt #{attempt}")

        # Fail on attempt 0 and 1, succeed on attempt 2+
        if attempt < 2:
            raise RuntimeError(f"Transient network error on attempt {attempt}")

        self.download_result = "data_batch_001.csv"  # artifact saved on success
        print(f"  Download succeeded on attempt {attempt}!")
        self.next(self.end)

    @step
    def end(self):
        print(f"Flow finished. Downloaded: {self.download_result}")

if __name__ == "__main__":
    RetryDemoFlow()
'''

with open('retry_flow.py', 'w') as f:
    f.write(flow_code)
print('retry_flow.py written.')


In [ ]:
# Run the retry flow — watch the retry messages in the output
!python retry_flow.py run --no-pylint 2>&1


**What just happened?**
- `@retry(times=3)` tells Metaflow to re-execute the step up to 3 extra times on any unhandled exception.
- **`METAFLOW_CURRENT_RETRY_COUNT`** is injected by Metaflow so the step knows which attempt it is on — great for backoff logic.
- The flow only moves to `end` once `flaky_download` succeeds — Metaflow handles all scheduling automatically.
- If all retries are exhausted the run is marked `failed`; artifacts from the successful attempt are preserved.


## Step 2 · @timeout — killing runaway steps

`@timeout` protects a pipeline from a step that hangs indefinitely. You can express the limit in seconds, minutes, or hours.

```python
@timeout(seconds=30)     # 30-second wall-clock limit
@timeout(minutes=5)      # 5-minute limit
@timeout(hours=2)        # 2-hour limit — for big training jobs
```

When the limit is hit, Metaflow raises `MetaflowTimeout` and the step is treated as a failure (subject to `@retry` if also decorated).


In [ ]:
timeout_flow_code = '''
import time
from metaflow import FlowSpec, step, timeout, retry

class TimeoutDemoFlow(FlowSpec):
    """Demonstrates @timeout cutting off a long-running step."""

    @step
    def start(self):
        self.next(self.fast_step, self.slow_step)

    @timeout(seconds=5)   # This step must finish within 5 seconds
    @step
    def fast_step(self):
        """Completes well within the timeout."""
        print("  fast_step: doing 1 second of work")
        time.sleep(1)
        self.fast_result = "done in 1s"
        self.next(self.join)

    @retry(times=1)         # Retry once after timeout
    @timeout(seconds=3)     # Very tight timeout to demonstrate the kill
    @step
    def slow_step(self):
        """Intentionally runs longer than the timeout on first attempt."""
        import os
        attempt = int(os.environ.get("METAFLOW_CURRENT_RETRY_COUNT", 0))
        if attempt == 0:
            print("  slow_step attempt 0: sleeping 10s (will timeout)")
            time.sleep(10)  # Will be killed by @timeout after 3s
        else:
            print("  slow_step attempt 1: doing fast work")
            time.sleep(1)   # Finishes in time on retry
        self.slow_result = f"completed on attempt {attempt}"
        self.next(self.join)

    @step
    def join(self, inputs):
        self.merge_artifacts(inputs)
        print(f"fast_result={self.fast_result}")
        print(f"slow_result={self.slow_result}")
        self.next(self.end)

    @step
    def end(self):
        print("All steps completed successfully.")

if __name__ == "__main__":
    TimeoutDemoFlow()
'''

with open('timeout_flow.py', 'w') as f:
    f.write(timeout_flow_code)
print('timeout_flow.py written.')


In [ ]:
!python timeout_flow.py run --no-pylint 2>&1


**What just happened?**
- `slow_step` was killed on attempt 0 because it exceeded the 3-second `@timeout`.
- **`@retry(times=1)`** kicked in and re-ran `slow_step` — this time the fast path succeeded.
- `@timeout` and `@retry` stack naturally: the timeout fires *per attempt*, not across all attempts.
- `merge_artifacts(inputs)` in the `join` step collects artifacts from both branches of the foreach.


## Step 3 · @catch — capturing exceptions as artifacts

`@catch` is the softer alternative to `@retry`. Instead of re-running the step, it:
1. Catches the exception
2. Stores it in the artifact named by `var`
3. Lets the flow continue

Use `@catch` for **optional enrichment steps** — e.g., fetching extra metadata — where failure is tolerable.

```python
@catch(var="enrich_error", print_exception=True)
@step
def enrich_data(self):
    # If this raises, error is stored in self.enrich_error and flow continues
    ...
```


In [ ]:
catch_flow_code = '''
from metaflow import FlowSpec, step, catch

class CatchDemoFlow(FlowSpec):
    """Demonstrates @catch allowing a flow to continue after a step failure."""

    @step
    def start(self):
        self.core_data = {"rows": 1000, "columns": 12}
        self.next(self.enrich_with_external_api)

    @catch(var="enrich_error", print_exception=True)
    @step
    def enrich_with_external_api(self):
        """Optional enrichment — we don't want the flow to die if this fails."""
        # Simulate a failing external API call
        raise ConnectionError("External enrichment API unavailable (503)")
        # This line would only run if no exception occurred:
        self.enrichment = {"geo": "US", "segment": "enterprise"}

    @step
    def report(self):
        # Check if enrichment succeeded or was caught
        if self.enrich_error is not None:
            print(f"  Enrichment failed (non-fatal): {self.enrich_error}")
            self.final_data = {**self.core_data, "enriched": False}
        else:
            self.final_data = {**self.core_data, **self.enrichment, "enriched": True}
        print(f"  Final data: {self.final_data}")
        self.next(self.end)

    @step
    def end(self):
        print("Flow completed despite enrichment failure.")

if __name__ == "__main__":
    CatchDemoFlow()
'''

with open('catch_flow.py', 'w') as f:
    f.write(catch_flow_code)
print('catch_flow.py written.')


In [ ]:
!python catch_flow.py run --no-pylint 2>&1


**What just happened?**
- `@catch(var="enrich_error")` stored the `ConnectionError` in `self.enrich_error` instead of crashing the flow.
- **`print_exception=True`** logs the traceback for debugging — you still *see* the error, just don't die on it.
- The `report` step checks `self.enrich_error` and branches logic — graceful degradation in pure Python.
- If the step had succeeded, `self.enrich_error` would be `None` and `self.enrichment` would be set.


## Step 4 · Combining all three decorators

Real production pipelines often stack all three on the same step — retry transient failures, time out truly stuck steps, and gracefully catch any remaining errors.

**Decorator stacking order matters:** Metaflow applies them bottom-up, so always write:
```python
@catch(var="err")     # outermost — last to apply
@retry(times=2)       # middle
@timeout(minutes=10)  # innermost — applies to each attempt
@step
def fetch_data(self):
    ...
```


In [ ]:
combined_flow_code = '''
import os
import time
from metaflow import FlowSpec, step, retry, timeout, catch

class ResilienceFlow(FlowSpec):
    """Production-grade flow combining @catch + @retry + @timeout on one step."""

    @step
    def start(self):
        print("Starting resilience demo...")
        self.next(self.fetch_model_weights)

    @catch(var="fetch_error", print_exception=True)  # outermost
    @retry(times=2)                                   # middle
    @timeout(seconds=8)                               # innermost — per attempt
    @step
    def fetch_model_weights(self):
        """Fetches model weights — might be slow, flaky, or completely unavailable."""
        attempt = int(os.environ.get("METAFLOW_CURRENT_RETRY_COUNT", 0))
        print(f"  fetch_model_weights attempt {attempt}")

        if attempt == 0:
            # Attempt 0: fast failure (network error)
            raise ConnectionError("S3 connection reset by peer")
        elif attempt == 1:
            # Attempt 1: succeeds
            time.sleep(1)
            self.weights_path = "s3://my-bucket/models/v3.pkl"
            print(f"  Weights fetched: {self.weights_path}")

    @step
    def train(self):
        if self.fetch_error is not None:
            print(f"  Skipping training — weights unavailable: {self.fetch_error}")
            self.model_path = None
        else:
            print(f"  Training with weights from {self.weights_path}")
            self.model_path = "output/model_v3_finetuned.pkl"
        self.next(self.end)

    @step
    def end(self):
        print(f"Pipeline complete. model_path={self.model_path}")

if __name__ == "__main__":
    ResilienceFlow()
'''

with open('resilience_flow.py', 'w') as f:
    f.write(combined_flow_code)
print('resilience_flow.py written.')


In [ ]:
!python resilience_flow.py run --no-pylint 2>&1


**What just happened?**
- **Attempt 0** raised `ConnectionError` — `@retry` scheduled attempt 1.
- **Attempt 1** succeeded — `self.weights_path` was set and `self.fetch_error` remained `None`.
- The `train` step found `fetch_error is None` and proceeded with real training logic.
- If all retries had failed, `@catch` would have stored the final exception and let `train` degrade gracefully.


## Step 5 · Deliberate failure testing

Good resilience engineering means **testing failure paths**, not just the happy path. Metaflow makes this easy: you can run flows with flags that force failures or inspect what happened after a real failure.

```bash
# Run and immediately inspect last run status
python flow.py run
python flow.py status         # shows last run result

# Inspect with the Client API (Day 9 topic — preview here)
from metaflow import Flow
run = Flow('ResilienceFlow').latest_run
print(run.successful)         # True/False
```

For unit-testing resilience logic in CI, inject failure via environment variables or monkey-patching — never rely on real external services in tests.


In [ ]:
# Inspect the last run of ResilienceFlow using the Client API
# (This works after the flow has been run at least once in the notebook)
from metaflow import Flow

try:
    flow = Flow('ResilienceFlow')
    run = flow.latest_run
    print(f"Run ID      : {run.id}")
    print(f"Successful  : {run.successful}")
    print(f"Finished at : {run.finished_at}")

    # Inspect each step's status
    for step in run:
        for task in step:
            print(f"  Step [{step.id}] Task [{task.id}] successful={task.successful}")
except Exception as e:
    print(f"Could not inspect run (has the flow been run yet?): {e}")


**What just happened?**
- `Flow('ResilienceFlow').latest_run` fetches the most recent run from Metaflow's local metadata store.
- Iterating `run → step → task` gives you granular success/failure per execution unit.
- **Even failed runs are preserved** — every retry attempt is a separate task, fully inspectable.
- This is a preview of the Client API deep-dive in Day 9.


## Step 6 · @retry with exponential backoff

`@retry` has a `minutes_between_retries` parameter for adding a delay between attempts — critical when hitting rate-limited APIs.

```python
@retry(times=3, minutes_between_retries=1)  # Wait 1 minute between retries
@step
def call_openai_api(self):
    ...
```

For more sophisticated backoff, implement it inside the step using `METAFLOW_CURRENT_RETRY_COUNT`:


In [ ]:
backoff_flow_code = '''
import os
import time
from metaflow import FlowSpec, step, retry

class BackoffFlow(FlowSpec):
    """Demonstrates manual exponential backoff inside a retried step."""

    @step
    def start(self):
        self.next(self.call_api)

    @retry(times=3)
    @step
    def call_api(self):
        attempt = int(os.environ.get("METAFLOW_CURRENT_RETRY_COUNT", 0))

        # Exponential backoff: 0, 1, 2 seconds on successive attempts
        # (Use larger values in production: 2**attempt seconds)
        backoff = attempt * 1
        if backoff > 0:
            print(f"  Backing off {backoff}s before attempt {attempt}...")
            time.sleep(backoff)

        # Fail on first 2 attempts to show the backoff
        if attempt < 2:
            raise IOError(f"Rate limited (attempt {attempt})")

        self.api_result = {"status": "ok", "attempt": attempt}
        print(f"  API call succeeded on attempt {attempt}: {self.api_result}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Result: {self.api_result}")

if __name__ == "__main__":
    BackoffFlow()
'''

with open('backoff_flow.py', 'w') as f:
    f.write(backoff_flow_code)

!python backoff_flow.py run --no-pylint 2>&1


**What just happened?**
- Each retry attempt slept progressively longer before trying again — mimicking real exponential backoff.
- **In production**, use `2 ** attempt` seconds (1s, 2s, 4s, 8s…) plus random jitter to avoid thundering-herd.
- `@retry(times=3, minutes_between_retries=1)` is simpler but gives fixed delays — manual code gives full control.
- Never add `time.sleep()` in a step without accounting for `@timeout` — they interact.


In [ ]:
# Challenge: Build a ResilientScraperFlow
#
# Implement a flow with these steps:
#   start -> fetch_urls -> process_each -> end
#
# Requirements:
#   - fetch_urls: use @retry(times=2) + @timeout(seconds=10)
#     Simulate success on the second attempt.
#
#   - process_each: use @catch(var="process_error")
#     Simulate a failure for one of the URLs and log the error in `end`.
#
#   - end: print how many URLs were processed successfully vs. failed.
#
# Scaffold:
scraper_code = '''
import os
from metaflow import FlowSpec, step, retry, timeout, catch

class ResilientScraperFlow(FlowSpec):

    @step
    def start(self):
        self.urls = [
            "https://example.com/page1",
            "https://example.com/page2",
            "https://bad-url.invalid/page3",
        ]
        self.next(self.fetch_urls)

    # TODO: add @retry(times=2) and @timeout(seconds=10)
    @step
    def fetch_urls(self):
        # TODO: simulate a transient failure on attempt 0, succeed on attempt 1
        self.fetched = self.urls
        self.next(self.process_each)

    # TODO: add @catch(var="process_error")
    @step
    def process_each(self):
        # TODO: raise an error for any URL containing "bad-url"
        self.results = [f"scraped: {u}" for u in self.fetched]
        self.next(self.end)

    @step
    def end(self):
        # TODO: print success count and process_error value
        pass

if __name__ == "__main__":
    ResilientScraperFlow()
'''

with open('scraper_flow.py', 'w') as f:
    f.write(scraper_code)
print('scraper_flow.py scaffold written — implement the TODOs above, then run:')
print('  !python scraper_flow.py run --no-pylint')


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `@retry(times=N)` | Re-runs the step on any exception, up to N extra attempts |
| `METAFLOW_CURRENT_RETRY_COUNT` | Injected env var — use for backoff or conditional logic |
| `@timeout(seconds=S)` | Kills the step after S wall-clock seconds; stacks with `@retry` |
| `@catch(var="e")` | Stores the exception in artifact `e`; flow continues regardless |
| Decorator order | `@catch` outermost, `@retry` middle, `@timeout` innermost, `@step` last |
| When to use `@catch` | Optional/enrichment steps where failure is acceptable |

> **Tip:** Always add `@retry` to steps that call external APIs or download data — transient failures are guaranteed in production.

---
## What's next
**Day 8** → Parameterized Flows: use `Parameter` and `JSONType` to make flows reusable across dev, staging, and prod configurations.

Mark Day 7 complete in your [tracker](../index.html).
